# Day 2 — Hidden states

This walkthrough starts with model loading and an inference-only smoke run. Extracting the first-token representation belongs to D2-02.

In [1]:
import sys
from pathlib import Path

import torch

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from transformers_learning import (
    DEFAULT_MODEL_NAME,
    load_model,
    load_tokenizer,
)

/home/makarlistkov/projects/transformers/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## D2-01 — Model inference

The tokenizer and model use one checkpoint name. `eval()` disables training-only behavior such as dropout, while `torch.no_grad()` prevents PyTorch from building a gradient graph for this inference call.

In [2]:
model_name = DEFAULT_MODEL_NAME
tokenizer = load_tokenizer(model_name)
model = load_model(model_name)
device = next(model.parameters()).device

print(f"Model name: {model_name}")
print(f"Device: {device}")
print(f"Training mode: {model.training}")
print(model)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3543.41it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model name: distilbert-base-uncased
Device: cpu
Training mode: False
DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)


In [3]:
text = "This movie was absolutely amazing!"
tokens = tokenizer(text, return_tensors="pt")
tokens = {name: tensor.to(device) for name, tensor in tokens.items()}

with torch.no_grad():
    gradients_enabled_during_inference = torch.is_grad_enabled()
    outputs = model(**tokens)

print(f"Output type: {type(outputs)}")
print(f"last_hidden_state shape: {tuple(outputs.last_hidden_state.shape)}")
print(f"Gradients enabled during inference: {gradients_enabled_during_inference}")

Output type: <class 'transformers.modeling_outputs.BaseModelOutput'>
last_hidden_state shape: (1, 8, 768)
Gradients enabled during inference: False
